# 🔵 Aula 11 — Agrupamento (Clustering)

## Descobrindo grupos e padrões sem utilizar rótulos

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de aprendizagem não supervisionada;
- Diferenciar aprendizagem supervisionada e não supervisionada;
- Compreender o conceito de agrupamento;
- Entender o funcionamento básico do algoritmo **K-Means**;
- Escolher variáveis para agrupamento;
- Normalizar variáveis antes do clustering;
- Aplicar K-Means com Python;
- Interpretar os grupos encontrados;
- Utilizar o método do cotovelo para auxiliar na escolha de `k`;
- Visualizar clusters;
- Analisar os centroides;
- Identificar aplicações de clustering no projeto individual.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Nesta aula queremos descobrir se os motores podem ser agrupados automaticamente de acordo com seu comportamento, **sem informar previamente ao algoritmo quais motores são "bons" ou "ruins"**.


# 🧠 1. O que é aprendizagem não supervisionada?

Até agora trabalhamos principalmente com análise dos dados.

Agora vamos começar a utilizar algoritmos capazes de encontrar padrões automaticamente.

### Aprendizagem supervisionada

Possuímos uma variável alvo.

Exemplo:

```text
Temperatura
Vibração
Corrente
      ↓
   Modelo
      ↓
Falha / Normal
```

### Aprendizagem não supervisionada

Não fornecemos uma resposta correta.

O algoritmo procura estruturas nos próprios dados.

```text
Temperatura
Vibração
Corrente
      ↓
  Algoritmo
      ↓
   Grupos
```

O **clustering** é uma das principais técnicas de aprendizagem não supervisionada.


# 🏭 2. Pergunta da indústria

Imagine que uma empresa possui centenas de motores.

Ela não possui uma classificação pronta dizendo:

```text
Motor A → tipo 1
Motor B → tipo 2
Motor C → tipo 1
```

Mas possui dados de sensores:

- temperatura;
- vibração;
- corrente;
- tensão;
- RPM.

A pergunta é:

> **Existem grupos de motores com comportamentos semelhantes?**

Essa é uma excelente pergunta para clustering.


# 🔵 3. O que é clustering?

Clustering significa **agrupamento**.

O algoritmo tenta colocar objetos semelhantes no mesmo grupo e objetos diferentes em grupos diferentes.

Imagine:

```text
        ● ●
      ● ● ●

                    ▲ ▲
                  ▲ ▲ ▲

      ■ ■
        ■ ■
```

O algoritmo não recebe:

```text
Grupo A
Grupo B
Grupo C
```

Ele precisa descobrir os grupos.

### Exemplos de aplicações

- segmentação de clientes;
- agrupamento de produtos;
- identificação de perfis de consumo;
- análise de comportamento;
- agrupamento de sensores;
- detecção exploratória de padrões;
- segmentação de documentos;
- análise de imagens.


# 📐 4. Distância e similaridade

Para agrupar objetos, precisamos definir o que significa serem semelhantes.

Uma ideia simples é utilizar distância.

Considere dois pontos:

```text
A = (2, 3)
B = (5, 7)
```

A distância euclidiana pode ser calculada por:

```text
d = √((x₂-x₁)² + (y₂-y₁)²)
```

Quanto menor a distância, mais semelhantes os pontos podem ser considerados em relação às variáveis utilizadas.

O K-Means utiliza a ideia de distância para construir seus grupos.


# 💻 5. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

print("Ambiente preparado!")

# 🏭 6. Criando dados de motores

Vamos criar três perfis fictícios de motores:

### Perfil A
Operação mais estável.

### Perfil B
Temperatura e vibração maiores.

### Perfil C
Corrente mais elevada.

O objetivo é verificar se o algoritmo consegue encontrar esses grupos sem receber os perfis como informação.


In [ ]:
n = 300

grupo_a = pd.DataFrame({
    "temperatura": np.random.normal(60, 3, n//3),
    "vibracao": np.random.normal(1.5, 0.25, n//3),
    "corrente": np.random.normal(11, 0.8, n//3)
})

grupo_b = pd.DataFrame({
    "temperatura": np.random.normal(78, 3, n//3),
    "vibracao": np.random.normal(3.2, 0.35, n//3),
    "corrente": np.random.normal(13, 0.8, n//3)
})

grupo_c = pd.DataFrame({
    "temperatura": np.random.normal(68, 3, n//3),
    "vibracao": np.random.normal(1.9, 0.25, n//3),
    "corrente": np.random.normal(16, 0.8, n//3)
})

df = pd.concat([grupo_a, grupo_b, grupo_c], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head()

Observe que não vamos informar ao K-Means qual é o grupo original.

Para o algoritmo, temos apenas os dados dos sensores.


# 🔎 7. Conhecendo os dados



In [ ]:
df.describe()

Vamos visualizar temperatura e vibração.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["vibracao"], df["temperatura"], alpha=0.6)
plt.title("Temperatura × Vibração")
plt.xlabel("Vibração")
plt.ylabel("Temperatura (°C)")
plt.show()

Talvez seja possível perceber alguns grupos visualmente.

Mas queremos que o algoritmo encontre esses grupos.


# ⚙️ 8. O algoritmo K-Means

O K-Means funciona, de forma simplificada, assim:

```text
1. Escolhemos K
       ↓
2. Criamos K centroides
       ↓
3. Cada ponto é associado ao centroide mais próximo
       ↓
4. Os centroides são recalculados
       ↓
5. Repetimos o processo
       ↓
6. Grupos finais
```

O **K** representa a quantidade de grupos desejados.

Exemplo:

```python
KMeans(n_clusters=3)
```

significa que queremos encontrar **3 clusters**.


# 📏 9. Por que normalizar?

Imagine duas variáveis:

```text
Temperatura → aproximadamente 60 a 80
Corrente    → aproximadamente 10 a 18
```

Neste caso a diferença de escala não é tão grande.

Mas imagine:

```text
Temperatura → 60
Pressão     → 100000
```

A variável pressão poderia dominar o cálculo de distância.

Por isso, geralmente normalizamos os dados antes do clustering.


In [ ]:
X = df[["temperatura", "vibracao", "corrente"]]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]

O `StandardScaler` transforma as variáveis para uma escala comparável, centrada em média 0 e desvio padrão 1.


# 🔵 10. Primeiro clustering

Vamos pedir ao K-Means para encontrar 3 grupos.


In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

df["cluster"] = clusters

df.head()

Agora o algoritmo criou uma nova coluna:

```text
cluster
```

Os números:

```text
0
1
2
```

são apenas identificadores dos grupos.

**Cluster 2 não significa necessariamente "melhor" que cluster 1.**


# 📊 11. Quantidade de registros por cluster



In [ ]:
df["cluster"].value_counts().sort_index()

Vamos visualizar os grupos.


In [ ]:
for cluster in sorted(df["cluster"].unique()):
    dados = df[df["cluster"] == cluster]
    plt.scatter(
        dados["vibracao"],
        dados["temperatura"],
        alpha=0.6,
        label=f"Cluster {cluster}"
    )

plt.title("Clusters — Temperatura × Vibração")
plt.xlabel("Vibração")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.show()

Agora o algoritmo encontrou grupos.

Mas ainda precisamos entender **o que cada grupo representa**.


# 📋 12. Analisando os centroides

O K-Means possui um centroide para cada grupo.

Vamos recuperar os centroides na escala original.


In [ ]:
centroides = scaler.inverse_transform(kmeans.cluster_centers_)

centroides_df = pd.DataFrame(
    centroides,
    columns=["temperatura", "vibracao", "corrente"]
)

centroides_df

Esses valores ajudam a criar uma interpretação.

Por exemplo:

```text
Cluster 0
Temperatura média: ...
Vibração média: ...
Corrente média: ...
```

A partir disso podemos construir um perfil do grupo.


# 🧠 13. Perfil dos clusters

Outra forma de analisar é agrupar a própria base.


In [ ]:
perfil_clusters = (
    df.groupby("cluster")[["temperatura", "vibracao", "corrente"]]
      .mean()
      .round(2)
)

perfil_clusters

Agora tente interpretar cada cluster.

Exemplo:

> O cluster X apresenta temperatura e vibração elevadas, podendo representar um grupo de motores que merece investigação.

**A interpretação é responsabilidade do analista.**

O algoritmo encontra os grupos; o especialista precisa contextualizá-los.


# 📐 14. Método do cotovelo

Como saber quantos clusters devemos utilizar?

Uma técnica exploratória bastante conhecida é o **método do cotovelo**.

O K-Means possui uma medida chamada `inertia_`.

Ela representa, de forma simplificada, a soma das distâncias dos pontos até seus respectivos centroides.

Vamos testar vários valores de K.


In [ ]:
inertias = []

for k in range(1, 11):
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    modelo.fit(X_scaled)
    inertias.append(modelo.inertia_)

inertias

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertias, marker="o")
plt.title("Método do Cotovelo")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inércia")
plt.xticks(range(1, 11))
plt.show()

A ideia é procurar um ponto em que aumentar K deixa de produzir uma redução significativa na inércia.

Esse ponto pode indicar um valor razoável para K.

> O método do cotovelo é uma ferramenta de apoio, não uma regra absoluta.


# 🔬 15. Testando diferentes valores de K

Vamos comparar alguns valores.


In [ ]:
resultados_k = []

for k in [2, 3, 4, 5]:
    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = modelo.fit_predict(X_scaled)

    resultados_k.append({
        "k": k,
        "inertia": modelo.inertia_,
        "quantidade_grupos": len(np.unique(labels))
    })

pd.DataFrame(resultados_k)

A escolha do número de clusters deve considerar:

- comportamento dos dados;
- conhecimento do problema;
- método do cotovelo;
- tamanho dos grupos;
- interpretabilidade;
- objetivo da análise.


# 📊 16. Visualizando três variáveis

Temos três variáveis:

```text
temperatura
vibracao
corrente
```

Um gráfico 2D consegue mostrar apenas duas dimensões ao mesmo tempo.

Podemos escolher duas variáveis e analisar a terceira por outro recurso, como tamanho dos pontos.


In [ ]:
tamanho = (df["corrente"] - df["corrente"].min() + 1) * 20

for cluster in sorted(df["cluster"].unique()):
    dados = df[df["cluster"] == cluster]
    tamanhos = tamanho.loc[dados.index]

    plt.scatter(
        dados["vibracao"],
        dados["temperatura"],
        s=tamanhos,
        alpha=0.5,
        label=f"Cluster {cluster}"
    )

plt.title("Clusters com Corrente representada pelo tamanho")
plt.xlabel("Vibração")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.show()

A visualização pode ajudar, mas cuidado para não transformar o gráfico em algo difícil de interpretar.

Quando temos muitas variáveis, outras técnicas de redução de dimensionalidade podem ser utilizadas.


# ⚠️ 17. Cuidados com clustering

Clustering não significa automaticamente que encontramos "verdades" nos dados.

Alguns cuidados:

### 1. K escolhido incorretamente

Um K muito pequeno pode juntar grupos diferentes.

Um K muito grande pode criar grupos artificiais.

### 2. Variáveis inadequadas

Variáveis sem relação com o problema podem prejudicar o agrupamento.

### 3. Escalas diferentes

É importante verificar a necessidade de normalização.

### 4. Outliers

Valores extremos podem influenciar os centroides.

### 5. Interpretação

O algoritmo cria grupos matemáticos.

Cabe ao especialista verificar se esses grupos fazem sentido no mundo real.


# 📝 18. Exercícios

## Exercício 1 — Conceitos

Explique com suas palavras a diferença entre:

**aprendizagem supervisionada** e **não supervisionada**.


In [ ]:
# Sua resposta



## Exercício 2 — Variáveis

Utilize apenas:

```text
temperatura
vibracao
```

e execute um K-Means com `k=2`.

Observe os grupos.


In [ ]:
# Sua resposta



## Exercício 3 — K = 4

Execute o K-Means utilizando `k=4`.

Compare os resultados com `k=3`.


In [ ]:
# Sua resposta



## Exercício 4 — Centroides

Mostre os centroides do modelo com `k=4`.

Interprete cada grupo.


In [ ]:
# Sua resposta



## Exercício 5 — Método do cotovelo

Crie um gráfico de inércia para:

```text
K = 1 até 10
```

Qual valor de K parece razoável?
Justifique.


In [ ]:
# Sua resposta



## Exercício 6 — Normalização

Execute o K-Means sem utilizar `StandardScaler`.

Depois compare com o resultado utilizando normalização.

O resultado mudou?


In [ ]:
# Sua resposta



## Exercício 7 — Perfil dos grupos

Para o modelo escolhido, calcule a média de:

- temperatura;
- vibração;
- corrente;

por cluster.

Depois descreva o perfil de cada grupo.


In [ ]:
# Sua resposta



## Exercício 8 — Visualização

Crie um gráfico de dispersão mostrando os clusters.

Utilize:

```text
X = temperatura
Y = corrente
```


In [ ]:
# Sua resposta



## Exercício 9 — Investigação industrial

Qual cluster apresenta:

- maior temperatura média?
- maior vibração média?
- maior corrente média?

Esse cluster merece investigação?
Justifique.


In [ ]:
# Sua resposta



## Exercício 10 — Reflexão

Imagine que o algoritmo encontrou três clusters:

```text
Cluster 0 → baixa temperatura / baixa vibração
Cluster 1 → alta temperatura / alta vibração
Cluster 2 → alta corrente / temperatura moderada
```

Proponha uma interpretação para cada grupo.


In [ ]:
# Sua resposta



# 🚀 19. Desafio — Clustering no seu projeto

Agora vamos transferir o conceito para o seu projeto individual.

### Etapa 1 — Escolha das variáveis

Escolha pelo menos **2 variáveis numéricas**.

### Etapa 2 — Preparação

Verifique:

- valores ausentes;
- valores extremos;
- escala das variáveis.

### Etapa 3 — Clustering

Teste pelo menos:

```text
K = 2
K = 3
K = 4
K = 5
```

### Etapa 4 — Escolha

Utilize o método do cotovelo e sua compreensão do problema para escolher um K.

### Etapa 5 — Interpretação

Descreva os grupos encontrados.

### Entrega

Você deverá apresentar:

```text
1. Problema
2. Variáveis escolhidas
3. Preparação dos dados
4. K testados
5. Gráfico do cotovelo
6. Modelo escolhido
7. Perfil dos clusters
8. Interpretação
9. Conclusão
```


In [ ]:
# Desenvolva o clustering do seu projeto aqui.



# 🏭 20. Aplicação no projeto didático

No exemplo dos motores, poderíamos chegar a algo como:

```text
CLUSTER 0
↓
Motores com temperatura e vibração baixas
↓
Perfil operacional estável

CLUSTER 1
↓
Motores com temperatura e vibração elevadas
↓
Perfil que merece investigação

CLUSTER 2
↓
Motores com corrente elevada
↓
Possível comportamento elétrico diferenciado
```

Mas atenção:

> **Esses nomes não são fornecidos pelo algoritmo.**

Nós interpretamos os grupos depois de analisar seus centroides e características.

Esse é um ponto fundamental da Mineração de Dados:

**o algoritmo encontra padrões; o analista transforma padrões em conhecimento.**


# 📌 21. Checklist da Aula

- [ ] Entendo aprendizagem não supervisionada;
- [ ] Entendo o conceito de clustering;
- [ ] Sei explicar a ideia do K-Means;
- [ ] Entendo o significado de K;
- [ ] Sei preparar as variáveis;
- [ ] Sei utilizar `StandardScaler`;
- [ ] Sei aplicar `KMeans`;
- [ ] Sei analisar os clusters;
- [ ] Sei analisar centroides;
- [ ] Sei utilizar o método do cotovelo;
- [ ] Sei comparar diferentes valores de K;
- [ ] Sei interpretar grupos;
- [ ] Consigo aplicar clustering ao meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 3  → Pandas
Aula 4  → Limpeza
Aula 5  → ETL
Aula 6  → Web Scraping
Aula 7  → Banco de Dados + SQL
Aula 8  → Análise Exploratória
Aula 9  → Amostragem + Balanceamento
Aula 10 → Visualização
Aula 11 → Clustering
```

Agora já conseguimos:

```text
OBTER DADOS
     ↓
PREPARAR
     ↓
EXPLORAR
     ↓
VISUALIZAR
     ↓
ENCONTRAR GRUPOS
```

Na próxima aula vamos avançar para outro grande problema de Mineração de Dados:

> 🎯 **Classificação — utilizando dados conhecidos para prever uma categoria.**

Vamos começar a trabalhar com **aprendizagem supervisionada**.
